## Notebook13

### Setup

Run all of the following before starting the notebook.

In [ ]:
! wget -q -nc https://raw.githubusercontent.com/taylor-arnold/fds2/refs/heads/main/funs.py

In [ ]:
import polars as pl
from plotnine import ggplot, aes
from polars import col as c

import funs

ub = "https://raw.githubusercontent.com/taylor-arnold/fds2/refs/heads/main/"

In [ ]:
play = pl.read_csv(ub + "data/shakespeare_plays.csv")
char = pl.read_csv(ub + "data/shakespeare_characters.csv")
line = pl.read_csv(ub + "data/shakespeare_lines.csv.gz")
metro = pl.read_csv(ub + "data/paris_metro_stops.csv")

### Questions

Today is about the shape of data: what a row is, what a key is, and what makes a table
tidy. We use two datasets.

The first is the Folger Shakespeare Library's edition of the plays, stored as three
tables. `play` has one row per play (37 of them), with a `play_id` and a `title`.
`char` has the characters, with a `character_id`, the `play_id`, and a `name`. `line`
has one row per printed line (80,592 of them), with the `play_id`, the `character_id`
of the speaker, the `act`, the `scene`, a `line_number` such as `"3.1.64"`, and the
`text`.

The second is a file of stops on the Paris Métro, `metro`, with a station `name`, the
`line` number, the `line_color`, and the `lon` and `lat` of the stop.

Unless a question says otherwise, just print the result of each block; do not save it
to a variable.

1. Start with a warm-up. Count the number of lines in each play, using `play_id`, and
sort from the most lines to the fewest.

In [ ]:
(
    line
    .group_by(c.play_id)
    .agg(n_lines = pl.len())
    .sort(c.n_lines, descending=True)
)

2. Take the code from the previous question and join `play` onto it to get the full
title of each play.

In [ ]:
(
    line
    .group_by(c.play_id)
    .agg(n_lines = pl.len())
    .join(play, on=c.play_id)
    .sort(c.n_lines, descending=True)
)

*Richard III* has the most lines, 3,592 of them.

3. The table `char` looks like it has one row per character. If that is true, then
`character_id` should never repeat. Count the number of rows for each `character_id`
and sort from the most rows to the fewest.

In [ ]:
(
    char
    .group_by(c.character_id)
    .agg(n = pl.len())
    .sort(c.n, descending=True)
)

Several ids appear four times, so `character_id` is not a primary key of this table.

4. Filter `char` to the rows where `character_id` is `"Bardolph_1H4"`.

In [ ]:
(
    char
    .filter(c.character_id == "Bardolph_1H4")
)

Bardolph appears in four different plays, and he has one row in each. So the unit of
observation of `char` is not a character. It is a *character × play*, and the primary
key is the pair of columns `character_id` and `play_id`.

5. Count the number of lines spoken by each `character_id` in the `line` table, and
sort from the most lines to the fewest.

In [ ]:
(
    line
    .group_by(c.character_id)
    .agg(n_lines = pl.len())
    .sort(c.n_lines, descending=True)
)

6. Now do the same count, but group by both `character_id` and `play_id`.

In [ ]:
(
    line
    .group_by(c.character_id, c.play_id)
    .agg(n_lines = pl.len())
    .sort(c.n_lines, descending=True)
)

The two answers disagree. Grouping by `character_id` alone adds up a character's
lines across every play they appear in. Richard III has 1,563 lines in the first table
but 1,148 in the second, because he also appears in two of the *Henry VI* plays. The
second table counts at the grain of the unit of observation.

7. On to tidy data. Filter `line` to *Hamlet* (`"ham"`), act 3, scene 1. Sort by
`line_number` and select the columns `line_number`, `character_id`, and `text`.

In [ ]:
(
    line
    .filter(c.play_id == "ham", c.act == 3, c.scene == 1)
    .sort(c.line_number)
    .select(c.line_number, c.character_id, c.text)
)

The lines come out in the order 3.1.1, 3.1.10, 3.1.100. The column `line_number` is
text, so it sorts alphabetically. It holds three variables packed into one cell (the
act, the scene, and the line within the scene), which breaks the first tidy rule: every
variable should have its own column.

8. Filter `line` to *Macbeth* (`"mac"`) and to the rows where `character_id` contains
the text `"WITCH"`. Count the lines for each `character_id`.

In [ ]:
(
    line
    .filter(c.play_id == "mac", c.character_id.str.contains("WITCH"))
    .group_by(c.character_id)
    .agg(n_lines = pl.len())
    .sort(c.n_lines, descending=True)
)

There are three witches, but four rows. The last one holds all three witches in a
single cell, which is how the Folger records the lines they speak together. That is
another cell holding more than one value.

9. Count the number of lines in each act of each play, grouping by `play_id` and `act`.

In [ ]:
(
    line
    .group_by(c.play_id, c.act)
    .agg(n_lines = pl.len())
    .sort(c.play_id, c.act)
)

This table has 185 rows, one per *play × act*. A table with one row per play and one
column per act would be easier to read, and we will learn how to build it with a
pivot in Chapter 9. We will also come back to the packed cells from questions 7 and 8.

10. Now the Paris Métro. Count the number of rows for each station `name` and sort from
the most rows to the fewest.

In [ ]:
(
    metro
    .group_by(c.name)
    .agg(n = pl.len())
    .sort(c.n, descending=True)
)

Châtelet has five rows because five lines stop there. The unit of observation is a
*station × line*, not a station.

11. The chapter says to record only data that describes the whole unit of observation.
Test `line_color`. Group by `line` and compute the number of stops and the number of
distinct colors with `c.line_color.n_unique()`.

In [ ]:
(
    metro
    .group_by(c.line)
    .agg(
        n_stops = pl.len(),
        n_colors = c.line_color.n_unique()
    )
    .sort(c.line)
)

Every line has exactly one color. `line_color` describes the line, not the
*station × line*, so the same color is written down again for every stop. It belongs in
a separate table with one row per line.

12. Draw the stops with longitude on the x-axis and latitude on the y-axis, colored by
`line`. Cast `line` to a string first, with `c.line.cast(pl.String)`, so that each line
gets its own color.

In [ ]:
(
    metro
    .with_columns(line = c.line.cast(pl.String))
    .ggplot(aes(c.lon, c.lat))
    .geom_point(aes(color=c.line))
)

13. The columns `lon_end` and `lat_end` were read in as text rather than numbers. Find
out why: filter `metro` to the rows where `lon_end` is equal to the text `"NA"`.

In [ ]:
(
    metro
    .filter(c.lon_end == "NA")
)

Eighteen rows use the text `"NA"` for a missing coordinate. That one choice turned the
whole `lon_end` and `lat_end` columns into text, even though every other value is a
number.

14. Last question, and it is a written one. The Métro has sixteen lines, including two
called 3bis and 7bis, but this file has only fourteen, because `line` was stored as an
integer. Design the tables this file should have had. For each table, give its unit of
observation, its primary key, and its columns.

**Answer**:
